# ShopSense 02: Clean the reviews and build passages

**Goal:** turn the raw reviews into clean tables and into short **passages**, the pieces of text that search will find and the answer model will quote.

**Reads:** the reviews and products parquet files in `data/processed/` (unchanged).
**Writes:** `reviews_clean.parquet`, `products_clean.parquet`, `passages.parquet` in `data/processed/`, and a funnel table in `reports/data_prep/`.

### What the audit (notebook 01) told us, and what we do about it
| Finding | Decision |
|---|---|
| 49,877 reviews, 24,564 products, **median 1 review per product** | Search over **all** reviews; give product-level answers only for products with **10+ reviews** (551 products). |
| 58% are 5 stars | Keep all ratings. Later, pros and cons need the 1 to 3 star reviews too. |
| Median 28 words, 75% under 56 words, 1% over 300 | **One review = one passage.** Only reviews over 300 words are split. |
| 12.5% under 10 words, 0.3% under 5 words | Drop under 5 words. Mark under 10 words as `is_short` (used to rank lower, not deleted). |
| 0.18% identical text | Drop exact duplicates (keep the first). |
| 43 users with 20+ reviews (3.3% of reviews) | Flag `heavy_reviewer`, do not delete. |
| 5 reviews look like instructions to an AI | Flag `looks_like_injection`, keep them. They are natural safety tests. |
| No date column | We cannot answer "recent" questions. We state this limit in the README. |
| Price empty for 81% of products | Do not use price. |

## 1. Setup

**What this does:** imports libraries, finds the project root and the input files, and makes the output folders.

In [ ]:
import html
import json
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"
OUT = ROOT / "reports" / "data_prep"
OUT.mkdir(parents=True, exist_ok=True)

REVIEWS_FILE = None
PRODUCTS_FILE = None
for p in sorted(PROCESSED.glob("*.parquet")):
    name = p.name.lower()
    if "review" in name and "clean" not in name and "passage" not in name:
        REVIEWS_FILE = p
    elif ("product" in name or "meta" in name) and "clean" not in name:
        PRODUCTS_FILE = p
print("reviews :", REVIEWS_FILE.name)
print("products:", PRODUCTS_FILE.name)

reviews : amazon_All_Beauty_reviews.parquet
products: amazon_All_Beauty_products.parquet


**What this does:** the rules in one place, so you can change a number and rerun.

In [ ]:
MIN_WORDS = 5              # drop reviews shorter than this
SHORT_WORDS = 10           # flag reviews shorter than this as is_short
MIN_REVIEWS_PRODUCT = 10   # product-level answers need at least this many clean reviews
HEAVY_USER = 20            # users with this many reviews or more get flagged
LONG_WORDS = 300           # split reviews longer than this
CHUNK_WORDS = 200          # words per piece when splitting
OVERLAP_WORDS = 40         # words shared between neighbouring pieces
INJECTION_PATTERN = r"ignore (?:all |the )?(?:previous|above|prior)|system prompt|disregard|you are now|as an ai|reveal your"


## 2. Load

**What this does:** loads reviews and products. Each review gets a permanent `review_id` from its row number **before** anything is dropped, so citations always point to the same review.

In [ ]:
reviews = pd.read_parquet(REVIEWS_FILE)
reviews = reviews.reset_index(drop=True)
reviews["review_id"] = ["r" + str(i).zfill(6) for i in range(len(reviews))]
products = pd.read_parquet(PRODUCTS_FILE)
print("reviews :", reviews.shape)
print("products:", products.shape)
print(reviews.columns.tolist())

reviews : (49877, 10)
products: (24564, 11)
['review_id', 'user_id', 'parent_asin', 'asin', 'rating', 'title', 'text', 'review_time', 'helpful_vote', 'verified_purchase']


## 3. Clean, step by step (with a count after each step)

**What this does:** starts the funnel table. After each rule we record how many reviews are left.

In [ ]:
funnel = []
funnel.append({"step": "start", "reviews_left": len(reviews)})
funnel

[{'step': 'start', 'reviews_left': 49877}]

**What this does:** cleans the text: removes HTML bits like `<br />` and Amazon media tags like `[[VIDEOID:...]]`, decodes symbols such as `&amp;`, and squeezes extra spaces. The original is kept in `text_raw`.

In [ ]:
reviews["text_raw"] = reviews["text"].fillna("").astype(str)
step1 = reviews["text_raw"].str.replace(r"<br\s*/?>", " ", regex=True)
step1 = step1.str.replace(r"\[\[(?:VIDEOID|ASIN|IMAGEID)[^\]]*\]\]", " ", regex=True)
step2 = step1.str.replace(r"<[^>]+>", " ", regex=True)
step3 = [html.unescape(t) for t in step2]
reviews["text"] = pd.Series(step3, index=reviews.index).str.split().str.join(" ")

reviews["title"] = reviews["title"].fillna("").astype(str).str.split().str.join(" ")
reviews["words"] = reviews["text"].str.split().str.len()
reviews[["text_raw", "text"]].head(3)

,text_raw,text
0,"You have to have the know how to work THIS Wig. It looks Wild and almost untrainable but if you have ANYKIND of Styling talent, it’s fix...","You have to have the know how to work THIS Wig. It looks Wild and almost untrainable but if you have ANYKIND of Styling talent, it’s fix..."
1,I love using this blush. So happy to find it on Amazon. Fast delivery.,I love using this blush. So happy to find it on Amazon. Fast delivery.
2,Staring with the basics.....the packaging was just as pictured above. Very colorful as well as sturdy. The material was made out of thic...,Staring with the basics.....the packaging was just as pictured above. Very colorful as well as sturdy. The material was made out of thic...


**What this does:** drops reviews with fewer than 5 words.

In [ ]:
reviews = reviews[reviews["words"] >= MIN_WORDS]
funnel.append({"step": f"drop under {MIN_WORDS} words", "reviews_left": len(reviews)})
funnel[-1]

{'step': 'drop under 5 words', 'reviews_left': 49737}

**What this does:** drops exact duplicate texts and keeps the first copy.

In [ ]:
reviews = reviews[~reviews["text"].duplicated(keep="first")]
funnel.append({"step": "drop exact duplicate text", "reviews_left": len(reviews)})
funnel[-1]

{'step': 'drop exact duplicate text', 'reviews_left': 49685}

**What this does:** drops reviews whose product is missing from the products table, or whose product has no title (we cannot cite a product without a name).

In [ ]:
products["title"] = products["title"].fillna("").astype(str).str.split().str.join(" ")
good_products = set(products.loc[products["title"] != "", "parent_asin"])
reviews = reviews[reviews["parent_asin"].isin(good_products)]
funnel.append({"step": "drop reviews with no named product", "reviews_left": len(reviews)})
funnel[-1]

{'step': 'drop reviews with no named product', 'reviews_left': 49677}

## 4. Flags (we mark, we do not delete)

**What this does:** adds yes/no columns: short review, contains link/email/phone, looks like an instruction to an AI, written by a heavy reviewer.

In [ ]:
reviews["is_short"] = reviews["words"] < SHORT_WORDS
reviews["has_contact_or_link"] = reviews["text"].str.contains(
    r"https?://|www\.|[\w.+-]+@[\w-]+\.[\w.]+|\b\d{3}[-. ]?\d{3}[-. ]?\d{4}\b", case=False, regex=True)
reviews["looks_like_injection"] = reviews["text"].str.contains(INJECTION_PATTERN, case=False, regex=True)

reviews_per_user = reviews["user_id"].value_counts()
heavy_users = set(reviews_per_user[reviews_per_user >= HEAVY_USER].index)
reviews["heavy_reviewer"] = reviews["user_id"].isin(heavy_users)

for flag in ["is_short", "has_contact_or_link", "looks_like_injection", "heavy_reviewer"]:
    print(f"{flag:22s} {int(reviews[flag].sum()):6d}  ({100 * reviews[flag].mean():.2f}%)")

is_short                 6095  (12.27%)
has_contact_or_link        36  (0.07%)
looks_like_injection        5  (0.01%)
heavy_reviewer           1660  (3.34%)


## 5. Products

**What this does:** builds the clean product table and adds, for each product, how many clean reviews it has and its average rating from those reviews. Products with 10+ reviews are marked `answerable` (enough evidence for a product-level answer).

In [ ]:
stats = reviews.groupby("parent_asin").agg(clean_reviews=("review_id", "count"), avg_rating_clean=("rating", "mean")).reset_index()
stats["avg_rating_clean"] = stats["avg_rating_clean"].round(2)

keep_cols = ["parent_asin", "title", "main_category", "average_rating", "rating_number", "store"]
keep_cols = [c for c in keep_cols if c in products.columns]
products_clean = products[keep_cols].merge(stats, on="parent_asin", how="inner")
products_clean["answerable"] = products_clean["clean_reviews"] >= MIN_REVIEWS_PRODUCT
print("products with clean reviews:", len(products_clean))
print("answerable (10+ reviews)   :", int(products_clean["answerable"].sum()))
products_clean.sort_values("clean_reviews", ascending=False).head(5)

products with clean reviews: 24493
answerable (10+ reviews)   : 548


,parent_asin,title,main_category,average_rating,rating_number,store,clean_reviews,avg_rating_clean,answerable
7954,B085BB7B1M,Salux Nylon Japanese Beauty Skin Bath Wash Cloth/towel (3) Blue Yellow and Pink,All Beauty,4.7,9101,SALUX,178,4.61,True
7103,B0BM4GX6TT,"Godefroy Tint Kit for Spot Coloring, Dark Brown",All Beauty,4.4,8691,Godefroy,116,4.07,True
18377,B01M1OFZOG,"Bed Head Curve Check Curling Wand for Tousled Waves and Texture, Jumbo Barrel",All Beauty,4.3,8144,Bed Head,112,3.71,True
9341,B07TK15BQQ,"R-NEU 200 Cotton Rounds, 100% Natural Turkish Cotton Makeup Remover and Facial Cleansing Cotton Round Pads, Waffle Textured Hypoallergen...",All Beauty,4.6,2484,R-NEU,108,4.34,True
6261,B00R1TAN7I,GranNaturals Boar Bristle Smoothing Hair Brush for Women and Men - Medium/Soft Bristles - Natural Wooden Large Flat Square Paddle Hairbr...,All Beauty,3.9,4199,GranNaturals,108,3.66,True


**What this does:** tells how many products and reviews each minimum would keep, so the 10-review choice is visible.

In [ ]:
rows = []
for limit in [1, 5, 10, 20, 50]:
    enough = products_clean[products_clean["clean_reviews"] >= limit]
    rows.append({"min_reviews": limit, "products": len(enough), "reviews": int(enough["clean_reviews"].sum()),
                 "reviews_pct": round(100 * enough["clean_reviews"].sum() / len(reviews), 1)})
tiers = pd.DataFrame(rows)
tiers.to_csv(OUT / "02_product_tiers.csv", index=False)
tiers

,min_reviews,products,reviews,reviews_pct
0,1,24493,49677,100.0
1,5,1724,17839,35.9
2,10,548,10566,21.3
3,20,143,5324,10.7
4,50,23,1856,3.7


## 6. Passages

**What this does:** builds the passages. Most reviews are short, so **one review = one passage**. A review longer than 300 words is cut into pieces of 200 words that overlap by 40 words, so no sentence is lost at a cut.

Each passage has: `passage_id` (like `r000123-0`), the review it came from, the product, the rating, `text` (what the user will see and what we quote) and `search_text` (the product title + review title + text, which is what search will index).

In [ ]:
title_of = dict(zip(products_clean["parent_asin"], products_clean["title"]))

passage_rows = []
for row in reviews.itertuples():
    words = row.text.split()
    if len(words) <= LONG_WORDS:
        pieces = [row.text]
    else:
        pieces = []
        start = 0
        while start < len(words):
            pieces.append(" ".join(words[start:start + CHUNK_WORDS]))
            if start + CHUNK_WORDS >= len(words):
                break
            start = start + CHUNK_WORDS - OVERLAP_WORDS

    for number, piece in enumerate(pieces):
        product_title = title_of[row.parent_asin]
        passage_rows.append({
            "passage_id": row.review_id + "-" + str(number),
            "review_id": row.review_id,
            "parent_asin": row.parent_asin,
            "product_title": product_title,
            "rating": row.rating,
            "review_title": row.title,
            "text": piece,
            "search_text": product_title + ". " + row.title + ". " + piece,
            "is_short": row.is_short,
            "heavy_reviewer": row.heavy_reviewer,
            "has_contact_or_link": row.has_contact_or_link,
            "looks_like_injection": row.looks_like_injection,
            "helpful_vote": getattr(row, "helpful_vote", 0),
            "verified_purchase": getattr(row, "verified_purchase", False),
        })

passages = pd.DataFrame(passage_rows)
print("reviews :", len(reviews))
print("passages:", len(passages), "(extra pieces come from long reviews)")

reviews : 49677
passages: 50524 (extra pieces come from long reviews)


## 7. Checks

**What this does:** checks the rules held: ids are unique, every passage has a product, no empty text, and shows passage length.

In [ ]:
assert passages["passage_id"].is_unique
assert passages["parent_asin"].isin(set(products_clean["parent_asin"])).all()
assert (passages["text"].str.strip() != "").all()

passages["passage_words"] = passages["text"].str.split().str.len()
print(passages["passage_words"].describe().round(1))
print("longest passage:", passages["passage_words"].max(), "words")

split_reviews = passages.groupby("review_id").size()
print("reviews that were split:", int((split_reviews > 1).sum()))

count    50524.0
mean        47.1
std         49.9
min          5.0
25%         15.0
50%         29.0
75%         58.0
max        300.0
Name: passage_words, dtype: float64
longest passage: 300 words
reviews that were split: 463


**What this does:** shows three random passages exactly as search will see them.

In [ ]:
for row in passages.sample(3, random_state=42).itertuples():
    print(row.passage_id, "| stars:", row.rating, "|", row.product_title[:60])
    print("   ", row.text[:250])
    print()

r024066-0 | stars: 5.0 | Gold & Snail Hydrogel Eye Patch (60 pcs) by Petitfee
    So cooling and relaxing! After trying them I immediately ordered another jar for a friend.

r015528-0 | stars: 5.0 | 24 Pieces Minimalist Dainty Hair Clip for Women, Tree Branch
    Be very gentle when taking it off.

r023526-0 | stars: 5.0 | Wahl Professional Adjustable 5 Star Standard Clipper Blade #
    I bought the wrong one the first time. The names need to be completely different to avoid having to send it back. I am very happy with this purchase and my Senior clipper.



## 8. Save

**What this does:** saves the three clean tables and the funnel. The original files are not touched.

In [ ]:
review_cols = ["review_id", "parent_asin", "user_id", "rating", "title", "text", "text_raw", "words",
               "is_short", "has_contact_or_link", "looks_like_injection", "heavy_reviewer"]
for extra in ["helpful_vote", "verified_purchase", "asin"]:
    if extra in reviews.columns:
        review_cols.append(extra)

reviews[review_cols].to_parquet(PROCESSED / "reviews_clean.parquet", index=False)
products_clean.to_parquet(PROCESSED / "products_clean.parquet", index=False)
passages.drop(columns=["passage_words"]).to_parquet(PROCESSED / "passages.parquet", index=False)

funnel_table = pd.DataFrame(funnel)
funnel_table["dropped"] = funnel_table["reviews_left"].shift(1) - funnel_table["reviews_left"]
funnel_table.to_csv(OUT / "02_funnel.csv", index=False)
funnel_table

,step,reviews_left,dropped
0,start,49877,NaN
1,drop under 5 words,49737,140.0
2,drop exact duplicate text,49685,52.0
3,drop reviews with no named product,49677,8.0


**What this does:** writes the rules and counts to a small JSON file, so anyone can see exactly how the clean data was made.

In [ ]:
manifest = {
    "rules": {"min_words": MIN_WORDS, "short_words": SHORT_WORDS, "min_reviews_product": MIN_REVIEWS_PRODUCT,
              "heavy_user": HEAVY_USER, "long_words": LONG_WORDS, "chunk_words": CHUNK_WORDS, "overlap_words": OVERLAP_WORDS},
    "counts": {"reviews_clean": len(reviews), "products_clean": len(products_clean),
               "products_answerable": int(products_clean["answerable"].sum()), "passages": len(passages)},
}
(OUT / "02_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))

{
  "rules": {
    "min_words": 5,
    "short_words": 10,
    "min_reviews_product": 10,
    "heavy_user": 20,
    "long_words": 300,
    "chunk_words": 200,
    "overlap_words": 40
  },
  "counts": {
    "reviews_clean": 49677,
    "products_clean": 24493,
    "products_answerable": 548,
    "passages": 50524
  }
}
